<a href="https://colab.research.google.com/github/ashishnathani2017/financegpt/blob/master/datacollectionandQLORAv3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!nvidia-smi | head -5
!pip -q install chromadb sentence-transformers pypdf

Wed Nov 25 17:41:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [4]:
from google.colab import files
import zipfile, pathlib
uploaded = files.upload()  # choose regulatory.zip
zipfile.ZipFile("regulatory.zip").extractall(".")
pdfs = sorted(pathlib.Path("regulatory").rglob("*.pdf"))
print(len(pdfs), "PDFs found")

Saving regulatory.zip to regulatory.zip
20 PDFs found


In [5]:
from pypdf import PdfReader
import re, json

def words(text):
    return re.sub(r"\s+", " ", text).strip().split(" ")

docs, chunks, stats = [], [], []
for p in pdfs:
    text = "\n".join((pg.extract_text() or "") for pg in PdfReader(p).pages)
    w = words(text)
    docs.append({"file": p.name, "chars": len(text), "words": len(w)})
    for i in range(0, len(w), 700):          # ~800 words, 100 overlap
        chunks.append({"doc": p.name, "text": " ".join(w[i:i+800])})
        if i + 800 >= len(w): break
print(f"{len(docs)} documents, {len(chunks)} chunks")
for d in docs: print(d)

20 documents, 584 chunks
{'file': '01MPR071020267B057FA4E71E4164A8B14EDC40197418.pdf', 'chars': 292610, 'words': 44884}
{'file': 'EDITEDTR101020267A1D86E2C75B46B79D8D065509673641.pdf', 'chars': 51085, 'words': 8692}
{'file': 'PR1266EE06BBCD7F5A4B2091FB59ADE87FE4A1.pdf', 'chars': 25120, 'words': 3909}
{'file': 'PR1361B0E4694B08CE42CDBEFE60069A43F7D0.pdf', 'chars': 30727, 'words': 4766}
{'file': 'PR5720E56F0A4611A404EACBD71F2207673A2.pdf', 'chars': 2945, 'words': 365}
{'file': 'PR62462AD650C8168481BAC17613CC54CFED8.pdf', 'chars': 2587, 'words': 348}
{'file': 'PR80907599DE5FD164918A49085C9D6270116.pdf', 'chars': 8866, 'words': 1293}
{'file': 'PR9257E0CB4D769F24658B0AB41428CA46F97.pdf', 'chars': 33835, 'words': 5295}
{'file': 'sebi_1996_Securities_and_Exchange_Board_of_India_Mutual_Funds_Regulations_1996_Last_amended_on_Novem_37f6cdb6.pdf', 'chars': 453853, 'words': 68885}
{'file': 'sebi_2018_Securities_and_Exchange_Board_of_India_Issue_of_Capital_and_Disclosure_Requirements_Regula_facf4e2

Build Index

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")
client = chromadb.Client()
col = client.create_collection("rbi_sebi", metadata={"hnsw:space": "cosine"})
emb = model.encode([c["text"] for c in chunks], batch_size=8, normalize_embeddings=True).tolist()
col.add(ids=[f"c{i}" for i in range(len(chunks))],
        documents=[c["text"] for c in chunks],
        metadatas=[{"doc": c["doc"]} for c in chunks],
        embeddings=emb)
print("indexed", col.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

indexed 584


Test Retrival

In [7]:
tests = [
  ("What is the voluntary lock-in facility for mutual fund folios?", "Voluntary"),
  ("Can mutual funds borrow money?", "Borrowing"),
  ("How should gold and silver held by mutual funds be valued?", "Gold"),
  ("What must social media posts by SEBI-regulated entities disclose?", "Social"),
  ("What is the RBI repo rate decision in October 2026?", "PR1266"),
  ("What did the Financial Stability Report say in June 2026?", "PR5720"),
]
hits = 0
for q, expect in tests:
    r = col.query(query_embeddings=model.encode([q], normalize_embeddings=True).tolist(), n_results=3)
    top = r["metadatas"][0][0]["doc"]
    ok = expect.lower() in top.lower()
    hits += ok
    print("OK " if ok else "MISS", q, "->", top[:70])
print(f"{hits}/{len(tests)} top hits")
json.dump({"documents": docs, "chunks": len(chunks), "top_hits": f"{hits}/{len(tests)}"},
          open("index_stats.json", "w"), indent=2)

OK  What is the voluntary lock-in facility for mutual fund folios? -> sebi_Mar_06_2026_Introduction_of_Voluntary_Lock-in_Debit_freeze_facili
MISS Can mutual funds borrow money? -> sebi_2026_Securities_and_Exchange_Board_of_India_Mutual_Funds_Regulati
OK  How should gold and silver held by mutual funds be valued? -> sebi_Feb_26_2026_Valuation_of_physical_Gold_and_Silver_held_by_mutual_
MISS What must social media posts by SEBI-regulated entities disclose? -> sebi_Feb_26_2026_Ease_of_Doing_Investment_EoDI_-_Disclosure_of_registe
OK  What is the RBI repo rate decision in October 2026? -> PR1266EE06BBCD7F5A4B2091FB59ADE87FE4A1.pdf
OK  What did the Financial Stability Report say in June 2026? -> PR5720E56F0A4611A404EACBD71F2207673A2.pdf
4/6 top hits


In [8]:
tests = [
  ("What is the voluntary lock-in facility for mutual fund folios?", "Voluntary"),
  ("Can mutual funds borrow money?", "Borrowing"),
  ("How should gold and silver held by mutual funds be valued?", "Gold"),
  ("What must social media posts by SEBI-regulated entities disclose?", "Social"),
  ("What is the RBI repo rate decision in October 2026?", "PR1266"),
  ("What did the Financial Stability Report say in June 2026?", "PR5720"),
]

hits1 = hits3 = 0
for q, expect in tests:
    r = col.query(query_embeddings=model.encode([q], normalize_embeddings=True).tolist(), n_results=3)
    tops = [m["doc"] for m in r["metadatas"][0]]
    ok1 = expect.lower() in tops[0].lower()
    ok3 = any(expect.lower() in t.lower() for t in tops)
    hits1 += ok1
    hits3 += ok3
    status = "top1" if ok1 else ("top3" if ok3 else "MISS")
    print(status, "|", q, "->", tops[0][:70])

print(f"top-1: {hits1}/{len(tests)} | top-3: {hits3}/{len(tests)}")

json.dump({"documents": docs, "chunks": len(chunks),
           "top1_hits": f"{hits1}/{len(tests)}", "top3_hits": f"{hits3}/{len(tests)}"},
          open("index_stats.json", "w"), indent=2)

top1 | What is the voluntary lock-in facility for mutual fund folios? -> sebi_Mar_06_2026_Introduction_of_Voluntary_Lock-in_Debit_freeze_facili
top3 | Can mutual funds borrow money? -> sebi_2026_Securities_and_Exchange_Board_of_India_Mutual_Funds_Regulati
top1 | How should gold and silver held by mutual funds be valued? -> sebi_Feb_26_2026_Valuation_of_physical_Gold_and_Silver_held_by_mutual_
MISS | What must social media posts by SEBI-regulated entities disclose? -> sebi_Feb_26_2026_Ease_of_Doing_Investment_EoDI_-_Disclosure_of_registe
top1 | What is the RBI repo rate decision in October 2026? -> PR1266EE06BBCD7F5A4B2091FB59ADE87FE4A1.pdf
top1 | What did the Financial Stability Report say in June 2026? -> PR5720E56F0A4611A404EACBD71F2207673A2.pdf
top-1: 4/6 | top-3: 5/6


# FinGPT-India: Month 2 — QLoRA fine-tuning (baseline FinLLM)

Run `datacollection.ipynb` first so the `chunks` list exists in the same session, or load chunks from a saved file.

Stand-in model: Qwen2.5-3B-Instruct (LLaMA 3.1 8B is too large for free Colab GPU). Say so in the report.

In [10]:
from google.colab import drive
drive.mount("/content/drive")

import json, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen2.5-3B-Instruct"
OUT = "/content/drive/MyDrive/fingpt-india/qlora-qwen2.5-3b"
MAX_LEN, EPOCHS, ACCUM, LR = 512, 2, 4, 2e-4

tok = AutoTokenizer.from_pretrained(BASE)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                       task_type="CAUSAL_LM",
                       target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

# Load examples; loss is computed on the answer tokens only
rows = [json.loads(l) for l in open("sft.jsonl", encoding="utf-8")]

def encode(row):
    msgs = row["messages"]
    prompt = tok.apply_chat_template(msgs[:1], add_generation_prompt=True, tokenize=False)
    full = prompt + msgs[1]["content"] + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"][:MAX_LEN]
    labels = ([-100] * len(p_ids) + f_ids[len(p_ids):])[:len(f_ids)]
    return f_ids, labels

data = [encode(r) for r in rows]
print("examples:", len(data))

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR)
model.train()
losses = []
for epoch in range(EPOCHS):
    random.Random(epoch).shuffle(data)
    for i, (ids, labels) in enumerate(data, start=1):
        x = torch.tensor([ids], device=model.device)
        y = torch.tensor([labels], device=model.device)
        loss = model(input_ids=x, labels=y).loss
        (loss / ACCUM).backward()
        losses.append(loss.item())
        if i % ACCUM == 0:
            opt.step(); opt.zero_grad()
    opt.step(); opt.zero_grad()
    print(f"epoch {epoch+1} mean loss: {sum(losses[-len(data):]) / len(data):.4f}")

model.save_pretrained(OUT)
tok.save_pretrained(OUT)
print("saved to", OUT, "| examples:", len(data), "| epochs:", EPOCHS,
      "| final mean loss:", round(sum(losses[-len(data):]) / len(data), 4))

Mounted at /content/drive


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import json, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
gen_model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")

random.seed(42)
sample = random.sample(chunks, 300)

PROMPT = """Read the passage and write ONE question a retail investor might ask that the passage answers, then the answer.
Use only facts in the passage. Reply exactly in this format:
Q: <question>
A: <answer>

Passage:
{passage}"""

def make_qa(passage):
    msgs = [{"role": "user", "content": PROMPT.format(passage=passage[:3000])}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(gen_model.device)
    out = gen_model.generate(**enc, max_new_tokens=200, do_sample=False)
    text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    if "Q:" in text and "A:" in text:
        q = text.split("Q:", 1)[1].split("A:", 1)[0].strip()
        a = text.split("A:", 1)[1].strip()
        return q, a
    return None, None

rows, failed = [], 0
for c in sample:
    q, a = make_qa(c["text"])
    if not q or not a or len(a) <= 20:
        failed += 1
        continue
    rows.append({"messages": [{"role": "user", "content": q},
                              {"role": "assistant", "content": a}],
                 "source": c["doc"]})

with open("sft.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"kept {len(rows)} of 300; discarded {failed} (format or length)")
del gen_model; torch.cuda.empty_cache()

In [ ]:
def make_qa(passage):
    msgs = [{"role": "user", "content": PROMPT.format(passage=passage[:3000])}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(gen_model.device)
    out = gen_model.generate(**enc, max_new_tokens=200, do_sample=False)
    text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    if "Q:" in text and "A:" in text:
        q = text.split("Q:", 1)[1].split("A:", 1)[0].strip()
        a = text.split("A:", 1)[1].strip()
        return q, a
    return None, None

rows, failed = [], 0
for c in sample:
    q, a = make_qa(c["text"])
    if not q or not a or len(a) <= 20:
        failed += 1
        continue
    rows.append({"messages": [{"role": "user", "content": q},
                              {"role": "assistant", "content": a}],
                 "source": c["doc"]})

with open("sft.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"kept {len(rows)} of 300; discarded {failed} (format or length)")
del gen_model; torch.cuda.empty_cache()



```
# This is formatted as code
```

## Step 2: Check 10 random examples by hand

Read each answer against its source. Record how many are not supported; these count as discards in the report.

In [ ]:
import random
lines = [json.loads(l) for l in open("sft.jsonl", encoding="utf-8")]
random.seed(7)
for r in random.sample(lines, min(10, len(lines))):
    print("SOURCE:", r["source"])
    print("Q:", r["messages"][0]["content"])
    print("A:", r["messages"][1]["content"][:300])
    print("-" * 60)


In [ ]:
import pathlib, re
from pypdf import PdfReader

def find(pdf_name, phrase, width=200):
    p = next(pathlib.Path("regulatory").rglob(pdf_name))
    text = " ".join((pg.extract_text() or "") for pg in PdfReader(p).pages)
    text = re.sub(r"\s+", " ", text)
    i = text.lower().find(phrase.lower())
    return text[max(0, i - 50): i + width] if i >= 0 else "NOT FOUND"


print(find("sebi_2018_*", "three days"))                              # #1
print(find("sebi_2026_*", "dissemination"))                           # #2
print(find("sebi_2018_*", "special bank account"))                    # #3
print(find("sebi_1996_*", "networth"))                                # #4
print(find("sebi_2018_*", "preferential"))                            # #5
print(find("sebi_2018_*", "company secretary"))                       # #6
print(find("sebi_2018_*", "underwriting"))                            # #7
print(find("sebi_1996_*", "remuneration"))                            # #8
print(find("sebi_2018_*", "ten thousand"))                            # #9
print(find("sebi_2026_*", "whistleblow"))                             # #10

In [ ]:
import pathlib, re
from pypdf import PdfReader

def find(pdf_name, phrase, width=200):
    p = next(pathlib.Path("regulatory").rglob(pdf_name))
    text = " ".join((pg.extract_text() or "") for pg in PdfReader(p).pages)
    text = re.sub(r"\s+", " ", text)
    i = text.lower().find(phrase.lower())
    return text[max(0, i - 50): i + width] if i >= 0 else "NOT FOUND"

In [11]:
print("#3", find("sebi_2018_*", "Investor Protection"))
print("#4", find("sebi_1996_*", "conflict of interest"))
print("#5", find("sebi_2018_*", "votes cast"))
print("#6", find("sebi_2018_*", "compliance officer"))

NameError: name 'find' is not defined

[link text](https://)## Step 3: QLoRA fine-tuning (GPU)

Mount Drive so the adapter survives a disconnect. Log the example count, epochs, learning rate, GPU and final loss for the report.

In [12]:
!ls -la /content/drive/MyDrive/fingpt-india/qlora-qwen2.5-3b

total 40011
drwx------ 2 root root     4096 Nov 20 05:06 .
drwx------ 4 root root     4096 Nov 20 05:06 ..
-rw------- 1 root root     1131 Nov 20 05:18 adapter_config.json
-rw------- 1 root root 29529752 Nov 20 05:18 adapter_model.safetensors
-rw------- 1 root root     2507 Nov 20 05:18 chat_template.jinja
-rw------- 1 root root     5202 Nov 20 05:18 README.md
-rw------- 1 root root      694 Nov 20 05:18 tokenizer_config.json
-rw------- 1 root root 11421892 Nov 20 05:18 tokenizer.json


In [15]:
!pip -q install -U trl peft bitsandbytes accelerate datasets

In [16]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-3B-Instruct"
OUT = "/content/drive/MyDrive/fingpt-india/qlora-qwen2.5-3b"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
#model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
# model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto",
#                                              torch_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto",
                                             dtype=torch.float16)
ds = load_dataset("json", data_files="sft.jsonl", split="train").remove_columns(["source"])

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

args = SFTConfig(output_dir=OUT, num_train_epochs=2, per_device_train_batch_size=2,
                 gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine",
                 warmup_steps=10, logging_steps=5, save_strategy="epoch",
                 fp16=False, report_to="none")
# for p in model.parameters():
#     if p.requires_grad:
#         p.data = p.data.float()
from peft import get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_cfg)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()
# trainer = SFTTrainer(model=model, args=args, train_dataset=ds,
#                      processing_class=tok, peft_config=peft_cfg)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok)
result = trainer.train()
trainer.save_model(OUT)
print("training loss:", result.training_loss, "| examples:", len(ds), "| epochs: 2")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

CHATBOT


In [30]:
!pip -q install -U chromadb sentence-transformers pypdf gradio transformers accelerate bitsandbytes peft trl datasets

:## Chatbot (Month 4): RAG + Qwen2.5-3B, optional QLoRA adapter

Needs `chatbot.py` uploaded in this session. Runtime: GPU (T4).

In [19]:
from google.colab import files, drive
import os, zipfile, pathlib
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/fingpt-india'
os.makedirs(PROJECT, exist_ok=True)

if not os.path.exists('chatbot.py'):
    files.upload()   # choose chatbot.py

if not os.path.exists('regulatory.zip') and not pathlib.Path(f'{PROJECT}/regulatory').exists():
    files.upload()   # choose regulatory.zip
if pathlib.Path('regulatory.zip').exists():
    zipfile.ZipFile('regulatory.zip').extractall(PROJECT)
print(len(list(pathlib.Path(f'{PROJECT}/regulatory').rglob('*.pdf'))), 'PDFs')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving chatbot.py to chatbot.py
20 PDFs


In [20]:
import importlib, chatbot
from chatbot import Config, Embedder, KnowledgeBase, Generator, FinGPTIndiaBot, build_ui, FALLBACK

ADAPTER_PATH = None   # set to f'{PROJECT}/sft-adapter' once QLoRA has been trained

cfg = Config(pdf_dir=f'{PROJECT}/regulatory', chroma_dir=f'{PROJECT}/chroma_db', adapter_path=ADAPTER_PATH)
embedder = Embedder(cfg.embed_model)
kb = KnowledgeBase(cfg, embedder)
kb.build()   # first run embeds all documents; later runs load the saved index

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

index already has 584 chunks (loaded from /content/drive/MyDrive/fingpt-india/chroma_db)


In [21]:
for q in ['Can mutual funds borrow money?', 'What is the RBI repo rate decision in October 2026?']:
    print(q)
    for h in kb.search(q, k=3):
        print(f"   {h['score']:.3f}  {h['doc'][:70]}")

Can mutual funds borrow money?
   0.670  sebi_2026_Securities_and_Exchange_Board_of_India_Mutual_Funds_Regulati
   0.638  sebi_Mar_13_2026_Borrowing_by_Mutual_Funds_1a0691b0.pdf
   0.620  sebi_1996_Securities_and_Exchange_Board_of_India_Mutual_Funds_Regulati
What is the RBI repo rate decision in October 2026?
   0.586  PR1266EE06BBCD7F5A4B2091FB59ADE87FE4A1.pdf
   0.570  PR80907599DE5FD164918A49085C9D6270116.pdf
   0.569  01MPR071020267B057FA4E71E4164A8B14EDC40197418.pdf


In [31]:
import pandas as pd, numpy as np, re, json, pathlib
EVENTS = pd.read_csv(f'{PROJECT}/market/events.csv', parse_dates=['start_date', 'end_date'])

def find_event(q):
    ql = q.lower()
    for _, e in EVENTS.iterrows():
        if any(a.strip() and a.strip() in ql for a in str(e['aliases']).split('|')):
            return e
    return None

def event_answer(e):
    last = NIFTY.index.max()
    end = min(e['end_date'], last)
    before = close_on(e['start_date'] - pd.Timedelta(days=1))
    after = close_on(end)
    if not before or not after:
        return "No Nifty data for that event window in the file."
    pct = 100 * (after[1] - before[1]) / before[1]
    w = NIFTY.loc[e['start_date']:end, 'Close']
    low_day = w.idxmin()
    caveat = "" if e['status'] == 'verified' else " (Event dates not yet checked against a primary source.)"
    return (f"{e['name']}: Nifty closed at {before[1]:,.2f} on {before[0].date()}, "
            f"and at {after[1]:,.2f} on {after[0].date()}, a change of {pct:+.2f}%. "
            f"The lowest close in the window was {w[low_day]:,.2f} on {low_day.date()}. "
            f"Window: {e['start_date'].date()} to {end.date()}.{caveat} {SRC}")

_prev_answer = answer_with_prices
def answer_with_events(question, **kw):
    e = find_event(question)
    if e is not None:
        return {"answer": event_answer(e),
                "sources": [{"n": 1, "doc": "nifty50_2020_2026.csv; events.csv", "score": None}],
                "used_rag": False}
    return _prev_answer(question, **kw)
bot.answer = answer_with_events

NameError: name 'answer_with_prices' is not defined

In [32]:
print(bot.answer.__name__)

NameError: name 'bot' is not defined

In [34]:
!pip install -q -U "bitsandbytes>=0.46.1" peft accelerate
llm = Generator(cfg)
bot = FinGPTIndiaBot(cfg, kb, llm)
for q in ['Can mutual funds borrow money? Under what limits?', 'Which stock should I buy this week?']:
    r = bot.answer(q)
    print('Q:', q)
    print('A:', r['answer'])
    print('Sources:', r['sources'], '\n')

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [35]:
!ls /content/drive/MyDrive/fingpt-india/market/
print(bot.answer.__name__)


events.csv  nifty50_2020_2026.csv


NameError: name 'bot' is not defined

In [ ]:
# @title
print(bot.answer.__name__)import pandas as pd, re

MARKET = f'{PROJECT}/market/nifty50_2020_2026.csv'
NIFTY = pd.read_csv(MARKET, parse_dates=['Date'])
NIFTY = NIFTY[NIFTY['Date'] != pd.Timestamp('2020-01-01')]   # holiday row, skipped in the lookup
NIFTY = NIFTY.set_index('Date').sort_index()

DATE_RE = re.compile(r'(\d{4}-\d{2}-\d{2})')

def price_answer(question):
    m = DATE_RE.search(question)
    if not m:
        return None
    d = pd.Timestamp(m.group(1))
    row = NIFTY.loc[:d].tail(1)
    if row.empty:
        return "No Nifty data on or before that date in the file."
    day = row.index[0].date()
    close = float(row['Close'].iloc[0])
    note = "" if day == d.date() else f" (the last trading day on or before {m.group(1)})"
    return (f"Nifty 50 closing level on {day}: {close:,.2f}{note}. "
            "Source: Yahoo Finance ^NSEI, downloaded for this project. Historical data only; not investment advice.")

_rag_answer = bot.answer
def answer_with_prices(question, **kw):
    p = price_answer(question)
    if p:
        return {"answer": p, "sources": [{"n": 1, "doc": "nifty50_2020_2026.csv (Yahoo ^NSEI)", "score": None}],
                "used_rag": False}
    return _rag_answer(question, **kw)
bot.answer = answer_with_prices

print(price_answer("What was the Nifty close on 2020-03-23?"))
print(price_answer("What is the RBI repo rate in October 2026?"))   # None: no date match, goes to documents

Nifty 50 closing level on 2020-03-23: 7,610.25. Source: Yahoo Finance ^NSEI, downloaded for this project. Historical data only; not investment advice.
None


V2

In [36]:
import pandas as pd, re

NIFTY = pd.read_csv(f'{PROJECT}/market/nifty50_2020_2026.csv', parse_dates=['Date'])
NIFTY = NIFTY[NIFTY['Date'] != pd.Timestamp('2020-01-01')].set_index('Date').sort_index()  # holiday row skipped

DATE_RE = re.compile(r'\b(\d{4}-\d{2}-\d{2}|\d{2}-\d{2}-\d{4})\b')
MOVE_WORDS = re.compile(r'percent|%|change|fall|fell|drop|down|rise|rose|\bup\b|gain|loss|lost|profit|return', re.I)
SRC = "Source: Yahoo Finance ^NSEI, downloaded for this project. Historical figures only; not investment advice."

def find_dates(q):
    out = []
    for s in DATE_RE.findall(q):
        out.append(pd.to_datetime(s, dayfirst=(len(s) == 10 and s[2] == '-')))
    return sorted(set(out))

def close_on(d):
    row = NIFTY.loc[:d].tail(1)
    if row.empty:
        return None
    return row.index[0], float(row['Close'].iloc[0])

def price_answer(q):
    dates = find_dates(q)
    if not dates:
        return None
    if len(dates) == 1 and MOVE_WORDS.search(q):
        r = close_on(dates[0])
        p = close_on(r[0] - pd.Timedelta(days=1)) if r else None
        if not r or not p:
            return "No Nifty data for that date in the file."
        (day, close), (pday, pclose) = r, p
        pct = 100 * (close - pclose) / pclose
        verb = "rose" if pct >= 0 else "fell"
        return (f"The Nifty 50 closed at {close:,.2f} on {day.date()}, which {verb} {abs(pct):.2f}% "
                f"({close - pclose:+,.2f} points) from {pclose:,.2f} on {pday.date()}. {SRC}")
    if len(dates) == 2 and MOVE_WORDS.search(q):
        r1, r2 = close_on(dates[0]), close_on(dates[1])
        if not r1 or not r2:
            return "No Nifty data for one of those dates in the file."
        pct = 100 * (r2[1] - r1[1]) / r1[1]
        return (f"From the close on {r1[0].date()} ({r1[1]:,.2f}) to the close on {r2[0].date()} ({r2[1]:,.2f}), "
                f"the Nifty 50 moved {pct:+.2f}%. This is a historical calculation, not a recommendation. {SRC}")
    r = close_on(dates[0])
    if not r:
        return "No Nifty data on or before that date in the file."
    note = "" if r[0] == dates[0] else f" (last trading day on or before {dates[0].date()})"
    return f"Nifty 50 closing level on {r[0].date()}: {r[1]:,.2f}{note}. {SRC}"

if '_rag_answer' not in globals():
    _rag_answer = bot.answer
def answer_with_prices(question, **kw):
    p = price_answer(question)
    if p:
        return {"answer": p, "sources": [{"n": 1, "doc": "nifty50_2020_2026.csv (Yahoo ^NSEI)", "score": None}],
                "used_rag": False}
    return _rag_answer(question, **kw)
bot.answer = answer_with_prices

print(price_answer("how much percentage nifty was down on 2020-09-23"))
print(price_answer("what was the Nifty close on 2020-03-23"))

NameError: name 'bot' is not defined

In [ ]:
q = "what is nifty 50 value after iran-us war"
print("event:", find_event(q))
print("answer:", bot.answer(q, use_rag=True)["answer"])

In [ ]:
r = answer_with_events(q)
print(r["answer"])
print(bot.answer)

In [ ]:
bot.answer = answer_with_events
print(bot.answer.__name__)

In [37]:
ADVICE_RE = re.compile(r'\b(should i|buy|sell|invest|tips?|recommend|worth it|good stock)\b', re.I)
ADVICE_MSG = (
    "I can't tell you whether to buy, sell or hold any security. Personal investment advice in India "
    "must come from a SEBI-registered investment adviser, who has to know your circumstances and "
    "disclose their fees and conflicts of interest. I can explain RBI and SEBI rules from the "
    "documents, and give historical Nifty figures from the data I have, which are not a forecast."
)
_before_advice = bot.answer
def answer_with_advice_check(question, **kw):
    if ADVICE_RE.search(question):
        return {"answer": ADVICE_MSG, "sources": [], "used_rag": False}
    return _before_advice(question, **kw)
bot.answer = answer_with_advice_check
print(bot.answer.__name__)

NameError: name 'bot' is not defined

In [40]:
!pip install -q -U "bitsandbytes>=0.46.1" peft accelerate

In [41]:
import gradio as gr


theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.orange,
    secondary_hue=gr.themes.colors.green,
    neutral_hue=gr.themes.colors.gray,
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
).set(
    body_background_fill="#ffffff",
    button_primary_background_fill="#FF9933",
    button_primary_background_fill_hover="#e8851f",
    button_primary_text_color="#ffffff",
)

CSS = """
.gradio-container {max-width: 860px !important; margin: auto;}
#tricolour {height: 6px; border-radius: 3px; margin-bottom: 10px;
  background: linear-gradient(to right, #FF9933 33%, #ffffff 33% 66%, #138808 66%);
  border: 1px solid #e5e5e5;}
#title h1 {color: #138808; font-weight: 700; margin-bottom: 0;}
#title p {color: #555;}
.message.bot, .bot {background: #f7f7f8 !important; border-radius: 14px !important;}
.message.user, .user {background: #fff3e6 !important; border-radius: 14px !important;}
"""

def chat_fn(message, history, use_adapter):
    r = bot.answer(message, use_rag=True, use_adapter=use_adapter)
    text = r["answer"]
    if r.get("sources"):
        text += "\n\n---\n**Sources:** " + "; ".join(
            f"[{s['n']}] {s['doc']} (similarity {s['score']})" for s in r["sources"])
    return text

with gr.Blocks(theme=theme, css=CSS, title="FinGPT-India") as demo:
    gr.HTML('<div id="tricolour"></div>')
    gr.Markdown("# 🇮🇳 FinGPT-India\nYour guide to Indian finance and market history. "
                "Not investment advice.", elem_id="title")
    gr.ChatInterface(
        fn=chat_fn,
        additional_inputs=[gr.Checkbox(value=bot.llm.has_adapter, interactive=bot.llm.has_adapter,
                                       label="Use fine-tuned adapter")],
        examples=[["How much did Nifty fall on 2020-03-23?", True],
                  ["What happened to Nifty after the Russia-Ukraine war began?", True],
                  ["What does SEBI say about insider trading?", True],
                  ["Can I buy Tata Power now?", True]],
    )
demo.launch(share=True)

/tmp/ipykernel_391/158381682.py:35: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=CSS, title="FinGPT-India") as demo:


NameError: name 'bot' is not defined

In [ ]:
# demo = build_ui(bot)
# demo.queue().launch(share=True, debug=True)   # public gradio.live link; share only with people you intend